# Greedy filling baseline

Approximate a target mesh with N placements from an item catalog. Placements
can intersect, rotate, and scale uniformly within the allowed range.

This example preserves the original volume-coverage greedy search while using
the refactored item, construction, task, and loss interfaces.


In [ ]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

from mesh_to_housing import (
    ConstructionLoss, Cuboid, FittingTask, GeometryConfig, LossConfig, LossTerm,
    MeshLoader, OutsideVolumeLoss, RandomPlacementGreedy, TransformConstraints,
    UncoveredVolumeLoss, fit, resize_mesh,
)
from mesh_to_housing.geometry.volume import VoxelGrid
from mesh_to_housing.losses import evaluate_volume

pio.renderers.default = "plotly_mimetype+notebook"
root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "src/mesh_to_housing").is_dir())
mesh = resize_mesh(MeshLoader.load(root / "mesh-assets/dinosaur.dae"), 10.0, axis=2)
print(f"{len(mesh.vertices):,} vertices, {len(mesh.faces):,} faces; watertight: {mesh.is_watertight}")

## Target and budget

The target is resized to a height of 10 units. The catalog bar always measures
0.25 × 0.25 × 1 unit, independently of the target. Placements can scale uniformly
from 20% to 200%; `(1.0, 1.0)` would disable resizing.

Voxel pitch controls the volume approximation. `closing=1` seals small target
voxel gaps before filling the interior. Fine features and intentional openings
can be affected by this approximation.

In [ ]:
n_items = 150
bar = Cuboid("bar", dimensions=(0.25, 0.25, 1.0))
task = FittingTask(
    target=mesh,
    catalog=(bar,),
    max_items=n_items,
    constraints=TransformConstraints(scale_bounds=(0.2, 2.0)),
)
geometry = GeometryConfig(voxel_pitch=float(mesh.extents.max()) / 96, closing=1)
loss = ConstructionLoss(LossConfig(components={
    "uncovered_volume": LossTerm(UncoveredVolumeLoss(geometry), weight=1.0),
    "outside_volume": LossTerm(OutsideVolumeLoss(geometry), weight=1.0),
}))
volume = VoxelGrid.from_mesh(task.target, pitch=geometry.voxel_pitch, closing=geometry.closing)
print(f"Target: {len(volume.cells):,} occupied cells; voxel pitch: {volume.pitch:.5g}")

## Greedy coverage

Sample target voxel centers once, without replacement. Build a fixed candidate
pool from every sampled center, rotation, scale, and catalog item. Discard
candidates that do not cover any target voxel, and cache their geometry.

At each step, choose the candidate with the greatest reduction in the loss.
For the volume loss below, this is new target coverage minus new outside volume.
Equal gains prefer more new target coverage, then less new outside volume, then
the first candidate in the pool. Overlap is counted once through union occupancy.

Continue until the budget is filled, including through nonpositive gains.
Insufficient resolved candidates raise an error. The seed reproduces the pool
and selection. Memory grows with the number and size of cached candidates.


In [ ]:
construction = fit(
    task,
    algorithm=RandomPlacementGreedy(
        geometry=geometry,
        candidate_centers=256,
        scale_count=5,
        orientation_count=8,
    ),
    loss=loss,
    seed=42,
)
print(f"{len(construction)} items; selected scales: {np.unique([p.scale for p in construction]).round(3)}")

In [ ]:
def mesh_trace(model, color, opacity=1.0):
    x, y, z = model.vertices.T
    i, j, k = model.faces.T
    return go.Mesh3d(x=x, y=y, z=z, i=i, j=j, k=k,
                     color=color, opacity=opacity, flatshading=True)


def show_result(show_target=True):
    traces = [mesh_trace(construction.to_mesh(), "#b47752")]
    if show_target:
        traces.insert(0, mesh_trace(mesh, "#8fa6bc", opacity=0.18))
    fig = go.Figure(traces)
    fig.update_layout(template="plotly_white", height=650,
                      scene=dict(aspectmode="data", camera=dict(up=dict(x=0, y=0, z=1))),
                      margin=dict(l=0, r=0, t=20, b=0))
    fig.show()


show_result()

In [ ]:
show_result(show_target=False)

## Two volume metrics

We use two voxel-based metrics to measure how well the placed items match the target shape.

Let **T** be the target volume and **A** the union of all placed items on the same voxel grid.

**Outside** measures how much of the placed volume lies outside the target:

$$
\text{Outside}
=
\frac{\operatorname{volume}(A \setminus T)}
{\operatorname{volume}(A)}
$$

**Uncovered** measures how much of the target is not covered by any placed item:

$$
\text{Uncovered}
=
\frac{\operatorname{volume}(T \setminus A)}
{\operatorname{volume}(T)}
$$

Lower values are better for both metrics.

Both metrics are estimated on the voxel grid, so details smaller than a voxel may be missed. Evaluation currently uses the same grid resolution as fitting.

Outside-volume error is also used in [Wang et al. (2006)](https://www.microsoft.com/en-us/research/wp-content/uploads/2017/01/sphapprox.pdf). Our version measures the union of all placed items rather than summing the error of each primitive separately.

In [ ]:
metrics = evaluate_volume(volume, construction)
print(f"Outside volume:   {metrics['outside_fraction']*100:.2f}% of the construction")
print(f"Uncovered volume: {metrics['uncovered_fraction']*100:.2f}% of the target")
print("Loss components:", loss.components(task.target, construction))

## Next steps

Add other fitting methods and metrics.

The first metric change should replace uncovered **volume** with uncovered **surface**, so hollow
interiors are acceptable.